In [ ]:
# Space Efficiency Evaluation
# This notebook calculates artifact sizes for all methods

import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.font_manager import FontProperties, fontManager
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

# Load fonts
FONT_DIR = Path(__file__).parent / 'fonts' if '__file__' in dir() else Path('fonts')
for f in FONT_DIR.glob('*.otf'):
    fontManager.addfont(str(f))
FONT_NAME = 'Linux Libertine O'

OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

sns.set(
    context='paper',
    style='ticks',
    palette='deep',
    font=FONT_NAME,
    font_scale=1.8,
    rc={
        'mathtext.fontset': 'stix',
        'pdf.fonttype': 42,
        'ps.fonttype': 42,
        'axes.labelweight': 'normal',
    }
)

# Define constants
DATA_ROOT = Path('../..')

# Align with eval_retrieval.ipynb ordering and abbreviations
DATASET_ORDER = ['adventure_works', 'chembl', 'public_bi', 'fetaqa', 'fetaqapn', 'bird', 'chicago']
DATASET_LABELS = {
    'adventure_works': 'Ad.', 'bird': 'BD.', 'chembl': 'Ch.',
    'chicago': 'Cc.', 'fetaqa': 'FL.', 'fetaqapn': 'FM.',
    'public_bi': 'Pb.',
}

def get_dir_size(path: Path) -> int:
    """Get total size of directory in bytes."""
    if not path.exists():
        return 0
    total = 0
    for f in path.rglob('*'):
        if f.is_file():
            total += f.stat().st_size
    return total

def get_file_size(path: Path) -> int:
    """Get size of file in bytes."""
    if not path.exists():
        return 0
    return path.stat().st_size

print('Utilities loaded.')

In [ ]:
# 1. Our Method: td_cd_cs/bm25 + td_cd_cs/faiss
# Location: data/lake/indexes/{dataset}/td_cd_cs/{bm25,faiss}
# Excluded: raw/bm25 (not used), test_query_*/train_query_* (eval cache)

Saturn_sizes = {}
for ds in DATASET_ORDER:
    base = DATA_ROOT / 'data/lake/indexes' / ds
    bm25 = get_dir_size(base / 'td_cd_cs/bm25')
    faiss = get_dir_size(base / 'td_cd_cs/faiss')
    total = bm25 + faiss
    Saturn_sizes[ds] = total
    print(f'{DATASET_LABELS[ds]:4s} ({ds}): {total:,} bytes ({total / 1024 / 1024:.2f} MB) [bm25: {bm25 / 1024 / 1024:.1f}MB, faiss: {faiss / 1024 / 1024:.1f}MB]')

print(f'\nTotal: {sum(Saturn_sizes.values()):,} bytes ({sum(Saturn_sizes.values()) / 1024 / 1024:.2f} MB)')

In [ ]:
# 2. Pneuma: bm25_index + chroma_index only
# Location: ref/pneuma_work/{dataset}/indices/bm25_index + indices/chroma_index
# These are the actual retrieval index artifacts loaded at query time
# Excluded: summaries/ (intermediate LLM outputs used to build the index)
# Excluded: converted/ (raw data copy, not an index artifact)

pneuma_sizes = {}
for ds in DATASET_ORDER:
    base = DATA_ROOT / 'ref/pneuma_work' / ds
    bm25 = get_dir_size(base / 'indices/bm25_index')
    chroma = get_dir_size(base / 'indices/chroma_index')
    total = bm25 + chroma
    pneuma_sizes[ds] = total
    print(f'{DATASET_LABELS[ds]:4s} ({ds}): {total:,} bytes ({total / 1024 / 1024:.2f} MB) [bm25: {bm25 / 1024 / 1024:.1f}MB, chroma: {chroma / 1024 / 1024:.1f}MB]')

print(f'\nTotal: {sum(pneuma_sizes.values()):,} bytes ({sum(pneuma_sizes.values()) / 1024 / 1024:.2f} MB)')

In [ ]:
# 3. Birdie: inference-only artifacts from final checkpoint (MT5-base, ~580M params, float32)
# Location: ref/birdie/output/{dataset}/checkpoint-{highest}/
# Birdie fine-tunes an MT5-base model per dataset to generate table IDs
# Inference needs: model.safetensors + tokenizer files (config, spiece.model, etc.)
# Excluded: optimizer.pt, rng_state, scheduler.pt, training_args.bin (training-only)

BIRDIE_INFERENCE_FILES = {
    'model.safetensors', 'config.json', 'generation_config.json',
    'special_tokens_map.json', 'spiece.model', 'tokenizer_config.json',
}

birdie_sizes = {}
for ds in DATASET_ORDER:
    base = DATA_ROOT / 'ref/birdie/output' / ds
    if not base.exists():
        birdie_sizes[ds] = None
        print(f'{DATASET_LABELS[ds]:4s} ({ds}): N/A (not found)')
        continue

    checkpoints = sorted(base.glob('checkpoint-*'), key=lambda p: int(p.name.split('-')[1]))
    if not checkpoints:
        birdie_sizes[ds] = None
        print(f'{DATASET_LABELS[ds]:4s} ({ds}): N/A (no checkpoints)')
        continue

    final_ckpt = checkpoints[-1]
    total = sum(
        f.stat().st_size for f in final_ckpt.iterdir()
        if f.is_file() and f.name in BIRDIE_INFERENCE_FILES
    )
    birdie_sizes[ds] = total
    print(f'{DATASET_LABELS[ds]:4s} ({ds}): {total:,} bytes ({total / 1024 / 1024 / 1024:.2f} GB) [{final_ckpt.name}]')

print(f'\nTotal (valid): {sum(v for v in birdie_sizes.values() if v):,} bytes ({sum(v for v in birdie_sizes.values() if v) / 1024 / 1024 / 1024:.2f} GB)')

In [ ]:
# 4. Solo: FAISS index files only
# Location: ref/index/on_disk_index_{dataset}_rel_graph/
# Files: merged_index.ivfdata + populated.index
# Excluded: passages.jsonl (raw passage text for result display, analogous to original table data)

solo_sizes = {}
for ds in DATASET_ORDER:
    base = DATA_ROOT / 'ref/index' / f'on_disk_index_{ds}_rel_graph'
    if not base.exists():
        solo_sizes[ds] = None
        print(f'{DATASET_LABELS[ds]:4s} ({ds}): N/A (not found)')
        continue
    
    ivfdata = get_file_size(base / 'merged_index.ivfdata')
    index = get_file_size(base / 'populated.index')
    total = ivfdata + index
    solo_sizes[ds] = total
    print(f'{DATASET_LABELS[ds]:4s} ({ds}): {total:,} bytes ({total / 1024 / 1024 / 1024:.2f} GB) [ivfdata: {ivfdata / 1024 / 1024 / 1024:.2f}GB, index: {index / 1024 / 1024:.1f}MB]')

print(f'\nTotal (valid): {sum(v for v in solo_sizes.values() if v):,} bytes')

In [ ]:
# Summary table (using eval_retrieval order and abbreviations)
print('=' * 80)
print('SPACE EFFICIENCY COMPARISON (in MB)')
print('=' * 80)

labels = [DATASET_LABELS[ds] for ds in DATASET_ORDER]
df = pd.DataFrame({
    'Saturn': {DATASET_LABELS[ds]: Saturn_sizes[ds] / 1024 / 1024 for ds in DATASET_ORDER},
    'Pneuma': {DATASET_LABELS[ds]: pneuma_sizes[ds] / 1024 / 1024 for ds in DATASET_ORDER},
    'Birdie': {DATASET_LABELS[ds]: (birdie_sizes[ds] / 1024 / 1024 if birdie_sizes[ds] else None) for ds in DATASET_ORDER},
    'Solo': {DATASET_LABELS[ds]: (solo_sizes[ds] / 1024 / 1024 if solo_sizes[ds] else None) for ds in DATASET_ORDER},
})

print(df.round(2).to_string())
print()
print('Totals:')
print(df.sum().round(2).to_string())

print('\n\nFairness Notes:')
print('- Saturn: td_cd_cs/bm25 + td_cd_cs/faiss (BM25 index + FAISS vector index)')
print('- Pneuma: BM25 index + Chroma vector index (retrieval artifacts loaded at query time)')
print('- Birdie: MT5-base inference model (~580M params, float32) per dataset')
print('- Solo: FAISS index files only (merged_index.ivfdata + populated.index)')
print('- All methods: raw table data excluded (shared across all methods)')

In [ ]:
# Lollipop Chart: Space Efficiency + Description Length Distribution
# Left y-axis: artifact size (MB, log scale) - 4 methods lollipop
# Right y-axis: description length (chars per table) - Saturn vs Pneuma box whiskers
# All 6 items share x-positions with uniform offsets
import json
import re
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

METHODS = ['Saturn', 'Pneuma', 'Birdie', 'Solo']
METHOD_COLORS = {
    'Saturn': '#3C6E8F',   # teal blue
    'Pneuma': '#C08552',   # warm bronze
    'Birdie': '#6B5B95',   # royal purple
    'Solo': '#B94A48',     # muted red
}
# 6 items: Saturn_size, Pneuma_size, Birdie_size, Solo_size, Saturn_desc, Pneuma_desc
ALL_ITEMS = ['Saturn', 'Pneuma', 'Birdie', 'Solo', 'Saturn_desc', 'Pneuma_desc']

# --- Collect description lengths per dataset ---
desc_data = {}
for ds in DATASET_ORDER:
    ours_corpus = DATA_ROOT / 'data/lake/indexes' / ds / 'td_cd_cs/bm25/index/corpus.jsonl'
    ours_lengths = []
    if ours_corpus.exists():
        with open(ours_corpus) as f:
            for line in f:
                entry = json.loads(line)
                ours_lengths.append(len(entry['text']))

    pneuma_corpus = DATA_ROOT / 'ref/pneuma_work' / ds / 'indices/bm25_index/corpus.jsonl'
    pneuma_table_texts = {}
    if pneuma_corpus.exists():
        with open(pneuma_corpus) as f:
            for line in f:
                entry = json.loads(line)
                table_id = entry['metadata']['table']
                base_table = re.sub(r'_SEP_contents_SEP_(schema|row)-\d+$', '', table_id)
                pneuma_table_texts.setdefault(base_table, []).append(entry['text'])
    pneuma_lengths = [sum(len(t) for t in texts) for texts in pneuma_table_texts.values()]
    desc_data[ds] = {'Saturn': ours_lengths, 'Pneuma': pneuma_lengths}

# --- Build lollipop data ---
data_rows = []
for ds in DATASET_ORDER:
    label = DATASET_LABELS[ds]
    sizes = {
        'Saturn': Saturn_sizes[ds] / 1024 / 1024,
        'Pneuma': pneuma_sizes[ds] / 1024 / 1024,
        'Birdie': birdie_sizes[ds] / 1024 / 1024 if birdie_sizes[ds] else None,
        'Solo': solo_sizes[ds] / 1024 / 1024 if solo_sizes[ds] else None,
    }
    data_rows.append((label, sizes))

n_datasets = len(data_rows)
n_items = len(ALL_ITEMS)
spacing = 0.85
x = np.arange(n_datasets) * spacing
offsets = np.linspace(-0.32, 0.32, n_items)

fig, ax = plt.subplots(figsize=(5.0, 2.5))

# --- Left axis: Lollipop (first 4 positions) ---
for m_idx, method in enumerate(METHODS):
    color = METHOD_COLORS[method]
    xs_plot, ys_plot = [], []
    for d_idx, (label, sizes) in enumerate(data_rows):
        val = sizes[method]
        xpos = x[d_idx] + offsets[m_idx]
        if val is None:
            ax.scatter(xpos, 0.15, marker='x', color='#999999', s=20, zorder=5, linewidths=0.8)
            continue
        xs_plot.append(xpos)
        ys_plot.append(val)
        ax.vlines(xpos, 0.1, val, color=color, alpha=0.5, linewidth=2.5, zorder=2)
    ax.scatter(xs_plot, ys_plot, color=color, s=35, zorder=5, edgecolors='white', linewidths=0.3)

ax.set_yscale('log')
ax.set_ylim(0.1, 50000)
ax.set_ylabel('Profile & Index Size (MiB)', fontsize=10, labelpad=1)

ax.set_xticks(x)
ax.set_xticklabels([row[0] for row in data_rows], fontsize=10)
margin = 0.32 * 1.2 + 0.12
ax.set_xlim(x[0] - margin, x[-1] + margin)

ax.tick_params(axis='y', labelsize=10, direction='in', width=0.8)
ax.tick_params(axis='x', length=0, width=0.8)

# Vertical dividers
for i in range(n_datasets - 1):
    mid = (x[i] + x[i + 1]) / 2
    ax.axvline(x=mid, color='#DDDDDD', linewidth=0.5, zorder=1)

# --- Right axis: Box whiskers (positions 4 & 5) ---
ax2 = ax.twinx()
whisker_width = 0.06
box_width = 0.10

for d_idx, ds in enumerate(DATASET_ORDER):
    for desc_m_idx, method in enumerate(['Saturn', 'Pneuma']):
        lengths = desc_data[ds][method]
        if not lengths:
            continue
        arr = np.array(lengths)
        p25, p75 = np.percentile(arr, 25), np.percentile(arr, 75)
        med = np.median(arr)
        p5, p95 = np.percentile(arr, 5), np.percentile(arr, 95)

        color = METHOD_COLORS[method]
        xpos = x[d_idx] + offsets[4 + desc_m_idx]

        # Whisker: P5 to P95
        ax2.vlines(xpos, p5, p95, color=color, linewidth=1.8, alpha=0.5, zorder=3)
        ax2.hlines(p5, xpos - whisker_width / 2, xpos + whisker_width / 2,
                    color=color, linewidth=1.8, alpha=0.5, zorder=3)
        ax2.hlines(p95, xpos - whisker_width / 2, xpos + whisker_width / 2,
                    color=color, linewidth=1.8, alpha=0.5, zorder=3)
        # Box: P25 to P75
        rect = plt.Rectangle((xpos - box_width / 2, p25), box_width, p75 - p25,
                               facecolor=color, alpha=0.35, edgecolor=color,
                               linewidth=1.2, zorder=4)
        ax2.add_patch(rect)
        # Median line
        ax2.hlines(med, xpos - box_width / 2, xpos + box_width / 2,
                    color=color, linewidth=2.0, alpha=0.9, zorder=5)

ax2.set_yscale('log')
ax2.set_ylim(50, 500000)
ax2.set_ylabel('Table Profile Length (chars)', fontsize=10, labelpad=1)
ax2.tick_params(axis='y', labelsize=10, direction='in', width=0.8)

# --- Legend: above BD column ---
# BD is index 5 in DATASET_ORDER
legend_handles = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=METHOD_COLORS['Saturn'],
           markersize=5, label='Saturn'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=METHOD_COLORS['Pneuma'],
           markersize=5, label='Pneuma'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=METHOD_COLORS['Birdie'],
           markersize=5, label='Birdie'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=METHOD_COLORS['Solo'],
           markersize=5, label='Solo'),
    Patch(facecolor=METHOD_COLORS['Saturn'], alpha=0.25, edgecolor=METHOD_COLORS['Saturn'],
          linewidth=0.6, label='Saturn'),
    Patch(facecolor=METHOD_COLORS['Pneuma'], alpha=0.25, edgecolor=METHOD_COLORS['Pneuma'],
          linewidth=0.6, label='Pneuma'),
]

leg = ax.legend(handles=legend_handles, fontsize=9, loc='lower center', frameon=True,
          fancybox=False, edgecolor='#CCCCCC', ncol=6,
          bbox_to_anchor=(0.5, 0.94),
          handletextpad=0.3, labelspacing=0.25, columnspacing=0.5,
          borderpad=0.3)
leg.get_frame().set_alpha(1.0)
leg.set_zorder(20)

for spine in ax.spines.values():
    spine.set_linewidth(0.2)
ax.spines['top'].set_visible(False)
for spine in ax2.spines.values():
    spine.set_linewidth(0.2)
ax2.spines['top'].set_visible(False)

fig.subplots_adjust(right=0.88)

fig.savefig(OUTPUT_DIR / 'space_efficiency.pdf', bbox_inches='tight', pad_inches=0.02, dpi=300)
print("Saved to", OUTPUT_DIR / "space_efficiency.pdf")

In [ ]:
# Description Length Comparison: Ours vs Pneuma (per-table)
# Ours: each line in corpus.jsonl = one table description
# Pneuma: multiple entries per table (schema-N, row-N), merge by table ID
import json
import re

desc_stats = []

for ds in DATASET_ORDER:
    label = DATASET_LABELS[ds]

    # --- Ours: one entry per table ---
    ours_corpus = DATA_ROOT / 'data/lake/indexes' / ds / 'td_cd_cs/bm25/index/corpus.jsonl'
    ours_lengths = []
    if ours_corpus.exists():
        with open(ours_corpus) as f:
            for line in f:
                entry = json.loads(line)
                ours_lengths.append(len(entry['text']))

    # --- Pneuma: merge split entries back to per-table ---
    pneuma_corpus = DATA_ROOT / 'ref/pneuma_work' / ds / 'indices/bm25_index/corpus.jsonl'
    pneuma_table_texts = {}  # base_table_name -> list of text chunks
    if pneuma_corpus.exists():
        with open(pneuma_corpus) as f:
            for line in f:
                entry = json.loads(line)
                table_id = entry['metadata']['table']
                # Strip _SEP_contents_SEP_schema-N or _SEP_contents_SEP_row-N suffix
                base_table = re.sub(r'_SEP_contents_SEP_(schema|row)-\d+$', '', table_id)
                pneuma_table_texts.setdefault(base_table, []).append(entry['text'])
    pneuma_lengths = [sum(len(t) for t in texts) for texts in pneuma_table_texts.values()]

    # Stats
    for method, lengths in [('Ours', ours_lengths), ('Pneuma', pneuma_lengths)]:
        if lengths:
            arr = np.array(lengths)
            desc_stats.append({
                'Dataset': label,
                'Method': method,
                'n_tables': len(arr),
                'mean': arr.mean(),
                'median': np.median(arr),
                'std': arr.std(),
                'min': arr.min(),
                'max': arr.max(),
                'p25': np.percentile(arr, 25),
                'p75': np.percentile(arr, 75),
            })

    print(f'--- {label} ({ds}) ---')
    if ours_lengths:
        a = np.array(ours_lengths)
        print(f'  Ours:   {len(a):4d} tables, mean={a.mean():.0f}, median={np.median(a):.0f}, '
              f'std={a.std():.0f}, range=[{a.min()}, {a.max()}]')
    if pneuma_lengths:
        a = np.array(pneuma_lengths)
        print(f'  Pneuma: {len(a):4d} tables, mean={a.mean():.0f}, median={np.median(a):.0f}, '
              f'std={a.std():.0f}, range=[{a.min()}, {a.max()}]')

# Summary DataFrame
df_desc = pd.DataFrame(desc_stats)
print('\n=== Description Length Summary (characters per table) ===')
print(df_desc.to_string(index=False, float_format='{:.0f}'.format))